In [ ]:
!pip install -q transformers sentencepiece spacy gensim
!python -m spacy download en_core_web_sm -q

import nltk
# import nltk
nltk.download('tagsets_json')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package tagsets_json to /root/nltk_data...
[nltk_data]   Package tagsets_json is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:300], "...")

Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization i ...


**5.1.** Implement subword-level tokenization using a BPE-based tokenizer.

Tokenize the sentence into subword units. Display the generated tokens and their corresponding token IDs.

(Implement two separate codes: a. Using pretrained model and b. Without using pretrained model)



### 5.1 (a) Using a Pretrained BPE Tokenizer




In [ ]:
sentence = "Natural language processing is a branch of artificial intelligence."
print(sentence)

Natural language processing is a branch of artificial intelligence.


In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [ ]:
tokens = tokenizer.tokenize(sentence)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token IDs:", token_ids)

Tokens: ['Natural', 'Ġlanguage', 'Ġprocessing', 'Ġis', 'Ġa', 'Ġbranch', 'Ġof', 'Ġartificial', 'Ġintelligence', '.']
Token IDs: [35364, 3303, 7587, 318, 257, 8478, 286, 11666, 4430, 13]



### 5.1 (b) BPE Tokenization Without Using a Pretrained Model



In [ ]:
import re
from collections import Counter

# Use the loaded text as training corpus
corpus = text.lower()
words = re.findall(r'\b\w+\b', corpus)

# Represent each word as space-separated characters + end marker
word_freq = Counter(words)
vocab = {" ".join(list(word)) + " </w>": freq for word, freq in word_freq.items()}

print("Sample vocabulary (word -> frequency):")
for w, f in list(vocab.items())[:5]:
    print(w, "->", f)

Sample vocabulary (word -> frequency):
n a t u r a l </w> -> 6
l a n g u a g e </w> -> 12
p r o c e s s i n g </w> -> 7
i s </w> -> 14
a </w> -> 28


In [ ]:
def get_pair_counts(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] += freq
    return pairs

pair_counts = get_pair_counts(vocab)
print("Top 5 most frequent pairs:", pair_counts.most_common(5))

Top 5 most frequent pairs: [(('e', '</w>'), 178), (('s', '</w>'), 174), (('e', 'n'), 125), (('i', 'n'), 107), (('n', '</w>'), 105)]


In [ ]:
def merge_vocab(vocab, pair):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab

In [ ]:
num_merges = 50
learned_merges = []

for i in range(num_merges):
    pair_counts = get_pair_counts(vocab)
    if not pair_counts:
        break
    best_pair = pair_counts.most_common(1)[0][0]
    vocab = merge_vocab(vocab, best_pair)
    learned_merges.append(best_pair)

print("Total merges learned:", len(learned_merges))
print("First 10 merges:", learned_merges[:10])

Total merges learned: 50
First 10 merges: [('e', '</w>'), ('s', '</w>'), ('e', 'n'), ('i', 'n'), ('d', '</w>'), ('e', 'r'), ('o', 'r'), ('a', 'n'), ('t', 'h'), ('o', 'n')]


In [ ]:
def bpe_tokenize_word(word, merges):
    symbols = list(word) + ["</w>"]
    symbols = [s for s in symbols]
    word_str = " ".join(symbols)

    for pair in merges:
        bigram = " ".join(pair)
        replacement = "".join(pair)
        word_str = word_str.replace(bigram, replacement)

    return word_str.split()

sentence = "natural language processing is a branch of artificial intelligence"
sentence_tokens = []

for word in sentence.split():
    sentence_tokens.extend(bpe_tokenize_word(word, learned_merges))

print("Subword Tokens:", sentence_tokens)

Subword Tokens: ['n', 'at', 'u', 'r', 'al</w>', 'l', 'an', 'g', 'u', 'a', 'g', 'e</w>', 'pr', 'oc', 'es', 's', 'ing</w>', 'i', 's</w>', 'a</w>', 'b', 'r', 'an', 'ch', '</w>', 'of</w>', 'ar', 't', 'i', 'f', 'ic', 'i', 'al</w>', 'in', 't', 'e', 'l', 'l', 'i', 'g', 'en', 'c', 'e</w>']


In [ ]:
unique_tokens = sorted(set(sentence_tokens))
token_to_id = {token: idx for idx, token in enumerate(unique_tokens)}

token_ids = [token_to_id[tok] for tok in sentence_tokens]

print("Tokens:", sentence_tokens)
print("Token IDs:", token_ids)

Tokens: ['n', 'at', 'u', 'r', 'al</w>', 'l', 'an', 'g', 'u', 'a', 'g', 'e</w>', 'pr', 'oc', 'es', 's', 'ing</w>', 'i', 's</w>', 'a</w>', 'b', 'r', 'an', 'ch', '</w>', 'of</w>', 'ar', 't', 'i', 'f', 'ic', 'i', 'al</w>', 'in', 't', 'e', 'l', 'l', 'i', 'g', 'en', 'c', 'e</w>']
Token IDs: [21, 6, 29, 25, 3, 20, 4, 15, 29, 1, 15, 11, 24, 22, 13, 26, 19, 16, 27, 2, 7, 25, 4, 9, 0, 23, 5, 28, 16, 14, 17, 16, 3, 18, 28, 10, 20, 20, 16, 15, 12, 8, 11]


**5.2.** Implement subword-level tokenization using a SentencePiece tokenizer (same input data as 5.1).

Tokenize the sentence into subword units. Display the generated tokens and their corresponding token IDs.

(Implement two separate codes: a. Using pretrained model and b. Without using pretrained model)



### 5.2 (a) Using a Pretrained SentencePiece Tokenizer



In [ ]:
from transformers import T5Tokenizer

sp_tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
sentence = "Natural language processing is a branch of artificial intelligence."

sp_tokens = sp_tokenizer.tokenize(sentence)
sp_token_ids = sp_tokenizer.convert_tokens_to_ids(sp_tokens)

print("Tokens:", sp_tokens)
print("Token IDs:", sp_token_ids)

Tokens: ['▁Natural', '▁language', '▁processing', '▁is', '▁', 'a', '▁branch', '▁of', '▁artificial', '▁intelligence', '.']
Token IDs: [6869, 1612, 3026, 19, 3, 9, 6421, 13, 7353, 6123, 5]


### 5.2 (b) Training Our Own SentencePiece Model (Without Pretrained)

In [ ]:
with open('corpus_for_sp.txt', 'w', encoding='utf-8') as f:
    f.write(text)

print("Corpus saved for training.")

Corpus saved for training.


In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input='corpus_for_sp.txt',
    model_prefix='my_sp_model',
    vocab_size=300,
    model_type='bpe'
)

print("SentencePiece model trained.")

SentencePiece model trained.


In [ ]:
sp = spm.SentencePieceProcessor()
sp.load('my_sp_model.model')

print("Model loaded.")

Model loaded.


In [ ]:
sentence = "natural language processing is a branch of artificial intelligence"

my_tokens = sp.encode(sentence, out_type=str)
my_token_ids = sp.encode(sentence, out_type=int)

print("Tokens:", my_tokens)
print("Token IDs:", my_token_ids)

Tokens: ['▁natural', '▁language', '▁processing', '▁is', '▁a', '▁b', 'r', 'an', 'ch', '▁of', '▁a', 'r', 't', 'if', 'ic', 'i', 'al', '▁int', 'el', 'l', 'i', 'g', 'ence']
Token IDs: [233, 118, 178, 84, 7, 32, 261, 15, 210, 58, 7, 261, 260, 75, 46, 264, 21, 156, 61, 265, 264, 272, 153]


**5.3.** Implement Part-of-Speech (POS) tagging on a given text and display the unique token, POS tag, and description of each POS tag.

**Input sentence:** "The young student is reading an interesting book in the library."

(Implement two separate codes: a. Using spaCy and b. NLTK)

### 5.3 (a) POS Tagging Using spaCy

In [ ]:
pos_sentence = "The young student is reading an interesting book in the library."
print(pos_sentence)

The young student is reading an interesting book in the library.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(pos_sentence)

print(f"{'Token':<15}{'POS Tag':<10}{'Description'}")
for token in doc:
    print(f"{token.text:<15}{token.pos_:<10}{spacy.explain(token.pos_)}")

Token          POS Tag   Description
The            DET       determiner
young          ADJ       adjective
student        NOUN      noun
is             AUX       auxiliary
reading        VERB      verb
an             DET       determiner
interesting    ADJ       adjective
book           NOUN      noun
in             ADP       adposition
the            DET       determiner
library        NOUN      noun
.              PUNCT     punctuation


### 5.3 (b) POS Tagging Using NLTK

In [ ]:
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.help import upenn_tagset
import io
import contextlib

tokens_nltk = word_tokenize(pos_sentence)
tagged = pos_tag(tokens_nltk)

print(f"{'Token':<15}{'POS Tag':<10}{'Description'}")
for word, tag in tagged:
    # Capture the description that upenn_tagset normally prints
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        upenn_tagset(tag)
    description = buf.getvalue().split(":")[-1].strip().split("\n")[0]
    print(f"{word:<15}{tag:<10}{description}")

Token          POS Tag   Description
The            DT        determiner
young          JJ        adjective or numeral, ordinal
student        NN        noun, common, singular or mass
is             VBZ       verb, present tense, 3rd person singular
reading        VBG       verb, present participle or gerund
an             DT        determiner
interesting    JJ        adjective or numeral, ordinal
book           NN        noun, common, singular or mass
in             IN        preposition or conjunction, subordinating
the            DT        determiner
library        NN        noun, common, singular or mass
.              .         sentence terminator


**5.4.** Implement Part-of-Speech (POS) tagging on a given text and display the unique token, POS tag, description of each token, and frequency using spaCy.

(Input data: same input data as 5.1)

In [ ]:
doc_full = nlp(text)
print("Total tokens in corpus:", len(doc_full))

Total tokens in corpus: 1209


In [ ]:
from collections import Counter

token_freq = Counter(token.text for token in doc_full if token.is_alpha)

# Build a dictionary of unique token -> (POS tag, description)
token_info = {}
for token in doc_full:
    if token.is_alpha and token.text not in token_info:
        token_info[token.text] = (token.pos_, spacy.explain(token.pos_))

In [ ]:
print(f"{'Token':<20}{'POS Tag':<10}{'Frequency':<12}{'Description'}")
for token_text, (pos, desc) in token_info.items():
    freq = token_freq[token_text]
    print(f"{token_text:<20}{pos:<10}{freq:<12}{desc}")

Token               POS Tag   Frequency   Description
Natural             ADJ       2           adjective
language            NOUN      11          noun
processing          NOUN      7           noun
is                  AUX       14          auxiliary
a                   DET       21          determiner
branch              NOUN      1           noun
of                  ADP       22          adposition
artificial          ADJ       1           adjective
intelligence        NOUN      1           noun
that                PRON      4           pronoun
enables             VERB      1           verb
computers           NOUN      1           noun
to                  PART      15          particle
understand          VERB      4           verb
interpret           VERB      1           verb
and                 CCONJ     27          coordinating conjunction
generate            VERB      1           verb
human               ADJ       1           adjective
combines            VERB      1          

**5.5.** Write a Python program to train a Word2Vec CBOW model using the given text corpus. Display the vector representation of a selected word and find the three most similar words.

**Input corpus (`corpus.txt`):**
```
Natural language processing is interesting
Natural language processing is useful
I love machine learning
Machine learning is useful
Deep learning is a part of machine learning
Artificial intelligence is changing the world
Natural language processing uses machine learning
Deep learning and artificial intelligence are related
I study natural language processing
Machine learning helps artificial intelligence
```



In [ ]:
corpus_text = """Natural language processing is interesting
Natural language processing is useful
I love machine learning
Machine learning is useful
Deep learning is a part of machine learning
Artificial intelligence is changing the world
Natural language processing uses machine learning
Deep learning and artificial intelligence are related
I study natural language processing
Machine learning helps artificial intelligence"""

with open('corpus.txt', 'w', encoding='utf-8') as f:
    f.write(corpus_text)

print("corpus.txt created.")

corpus.txt created.


In [ ]:
with open('corpus.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

sentences = [line.lower().split() for line in lines if line.strip()]

print("Tokenized sentences:")
for s in sentences:
    print(s)

Tokenized sentences:
['natural', 'language', 'processing', 'is', 'interesting']
['natural', 'language', 'processing', 'is', 'useful']
['i', 'love', 'machine', 'learning']
['machine', 'learning', 'is', 'useful']
['deep', 'learning', 'is', 'a', 'part', 'of', 'machine', 'learning']
['artificial', 'intelligence', 'is', 'changing', 'the', 'world']
['natural', 'language', 'processing', 'uses', 'machine', 'learning']
['deep', 'learning', 'and', 'artificial', 'intelligence', 'are', 'related']
['i', 'study', 'natural', 'language', 'processing']
['machine', 'learning', 'helps', 'artificial', 'intelligence']


In [ ]:
from gensim.models import Word2Vec

# sg=0 means CBOW (Continuous Bag of Words)
cbow_model = Word2Vec(sentences, vector_size=50, window=2, min_count=1, sg=0)

print("CBOW model trained.")

CBOW model trained.


In [ ]:
word = "learning"
vector = cbow_model.wv[word]

print(f"Vector for '{word}':\n", vector)

Vector for 'learning':
 [-1.06855005e-03  4.69933148e-04  1.02008386e-02  1.80172808e-02
 -1.86104029e-02 -1.42386770e-02  1.29159437e-02  1.79444924e-02
 -1.00341877e-02 -7.52950506e-03  1.47660999e-02 -3.07230861e-03
 -9.05987713e-03  1.31118027e-02 -9.72144213e-03 -3.62110720e-03
  5.76059474e-03  1.98688661e-03 -1.65799800e-02 -1.88948642e-02
  1.46290399e-02  1.01473518e-02  1.35137467e-02  1.51792879e-03
  1.27044776e-02 -6.80534309e-03 -1.88327127e-03  1.15384720e-02
 -1.50461625e-02 -7.86929950e-03 -1.50221856e-02 -1.86574075e-03
  1.90685876e-02 -1.46321915e-02 -4.67193313e-03 -3.87651450e-03
  1.61498692e-02 -1.18692387e-02  1.00449470e-04 -9.50518623e-03
 -1.92147940e-02  1.00254593e-02 -1.75086837e-02 -8.78774561e-03
 -6.72474198e-05 -5.80182823e-04 -1.53198056e-02  1.92264281e-02
  9.96836554e-03  1.84633974e-02]


In [ ]:
similar_words = cbow_model.wv.most_similar(word, topn=3)

print(f"Most similar words to '{word}':")
for w, score in similar_words:
    print(w, "->", round(score, 4))

Most similar words to 'learning':
the -> 0.2706
are -> 0.2106
love -> 0.1861
